In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F 
from sklearn.preprocessing import StandardScaler


import numpy as np
import pandas as pd

In [7]:
device = torch.device(
    "cude" if torch.cuda.is_available() else "cpu"
)
device

device(type='cpu')

In [2]:
scholar_df = pd.DataFrame({
    "scholar_id": [1, 2, 3, 4],

    "research_interests": [
        "deep learning for flood prediction and hydrological modelling",
        "computer vision and medical image analysis",
        "structural health monitoring using machine learning",
        "remote sensing and climate change"
    ],

    "expertise": [
        "deep learning, hydrology, remote sensing",
        "computer vision, deep learning, medical imaging",
        "machine learning, structural engineering, sensors",
        "remote sensing, GIS, climate modelling"
    ],

    "department": [
        "Civil Engineering",
        "Computer Science",
        "Civil Engineering",
        "Civil Engineering"
    ],

    "university": [
        "IIT Kharagpur",
        "IIT Delhi",
        "IIT Bombay",
        "IIT Kharagpur"
    ],

    "country": [
        "India",
        "India",
        "India",
        "India"
    ],

    "publication_count": [
        12, 25, 18, 30
    ],

    "citation_count": [
        120, 450, 200, 600
    ],

    "years_experience": [
        2, 5, 3, 7
    ]
})

scholar_df

,scholar_id,research_interests,expertise,department,university,country,publication_count,citation_count,years_experience
0,1,deep learning for flood prediction and hydrolo...,"deep learning, hydrology, remote sensing",Civil Engineering,IIT Kharagpur,India,12,120,2
1,2,computer vision and medical image analysis,"computer vision, deep learning, medical imaging",Computer Science,IIT Delhi,India,25,450,5
2,3,structural health monitoring using machine lea...,"machine learning, structural engineering, sensors",Civil Engineering,IIT Bombay,India,18,200,3
3,4,remote sensing and climate change,"remote sensing, GIS, climate modelling",Civil Engineering,IIT Kharagpur,India,30,600,7


In [3]:
scholar_df.columns

Index(['scholar_id', 'research_interests', 'expertise', 'department',
       'university', 'country', 'publication_count', 'citation_count',
       'years_experience'],
      dtype='object')

In [4]:
numerical_columns = ["publication_count", "citation_count", "years_experience"]

In [ ]:
scaler = StandardScaler()
numeric_values = scaler.fit_transform(
    scholar_df[numerical_columns]
)

"""
ArithmeticErrorpublication_count   → mean 0, std 1
citation_count                     → mean 0, std 1
years_experience                   → mean 0, std 1
"""

In [6]:
numeric_values

array([[-1.35375921, -1.15799457, -1.1717002 ],
       [ 0.5488213 ,  0.55948052,  0.39056673],
       [-0.47564513, -0.74163697, -0.65094455],
       [ 1.28058303,  1.34015101,  1.43207802]])

In [10]:
numeric_tensor = torch.tensor(
    numeric_values,
    dtype=torch.float32,
    device=device
)
numeric_tensor.shape

torch.Size([4, 3])

In [11]:
class NumercalEncoder(nn.Module):
    def __init__(self, input_dim, output_dim = 32):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),

            nn.Linear(32, output_dim)
        )
    def forward(self, x):
        return self.network(x)

In [13]:
numerical_encoder = NumercalEncoder(input_dim=len(numerical_columns), output_dim=32).to(device)
numerical_encoder

NumercalEncoder(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
  )
)

In [14]:
numerical_embeddings = numerical_encoder(
    numeric_tensor
)
print(numerical_embeddings.shape)

torch.Size([4, 32])


In [ ]:
# combined_embedding = torch.cat(
#     [
#         text_embeddings,
#         categorical_embedding,
#         numeric_embedding
#     ],
#     dim=1
# )

# print(combined_embedding.shape)
# torch.Size([4, 512])

'''
                    Scholar
                       │
          ┌────────────┼────────────┐
          ↓            ↓            ↓
       Text         Categorical    Numeric
       384             160           32
          │            │             │
          └────────────┼─────────────┘
                       ↓
                    CONCAT
                       ↓
                      512
'''